# Notebook: `04. Bronze`
---
## Descrição

Esse notebook realiza a ingestão dos dados brutos nos volumes da camada `staging` para tabelas persistidas na camada bronze.

## Origem

- **yellowTaxiTripData**: `/Volumes/mvp/staging/yellow_taxi`;
- **greenTaxiTripData**: `/Volumes/mvp/staging/green_taxi`;
- **forHireVehicles**: `/Volumes/mvp/staging/fhv`
- **forHireVehiclesHighVolume**: `/Volumes/mvp/staging/fhvhv`
- **taxi_zone_lookup**: `/Volumes/mvp/staging/taxi_zones/taxi_zone_lookup.csv`
- **fhv_base_lookup**: `/Volumes/mvp/staging/misc/fhv_lookup.csv`
- **Feriados_US_NY**: `/Volumes/mvp/staging/misc/Feriados_US_NY.csv`
- **Inflacao_CPI_U**: `/Volumes/mvp/staging/misc/CPIAUCSL (2016 - 2026).csv`
- **Inflacao_CPI_T**: `/Volumes/mvp/staging/misc/CPITRNSL (20216-2026).csv`

## Destino

- **yellowTaxiTripData**: `mvp.bronze.yellowTaxiTripData`;
- **greenTaxiTripData**: `mvp.bronze.greenTaxiTripData`;
- **forHireVehicles**: `mvp.bronze.forHireVehicles`
- **forHireVehiclesHighVolume**: `mvp.bronze.forHireVehiclesHighVolume`
- **taxi_zone_lookup**: `mvp.bronze.taxi_zone_lookup`
- **fhv_base_lookup**: `mvp.bronze.fhv_base_lookup`
- **Feriados_US_NY**: `mvp.bronze.Feriados_US_NY`
- **Inflacao_CPI_U**: `mvp.bronze.Inflacao_CPI_U`
- **Inflacao_CPI_T**: `mvp.bronze.Inflacao_CPI_T`

Define o catálogo e schema usado.

In [0]:
%sql
USE mvp.bronze

Carrega bibliotecas necessárias.

In [0]:
from pyspark.sql.functions import col, lit, coalesce, regexp_extract
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, TimestampNTZType

**dedupe_airport_fee**: Resolve a duplicidade de grafia da coluna de taxa de aeroporto presente nos arquivos da TLC a partir de 2025, nos quais coexistem `airport_fee` (minúscula) e `Airport_fee` (maiúscula) no mesmo `.parquet`. Quando as duas colunas existem, une seus valores em uma única coluna `airport_fee` (usando o primeiro valor não nulo entre elas) e descarta as originais. Quando existe apenas a versão com maiúscula, apenas a renomeia para `airport_fee`. Quando existe apenas a versão em minúscula, o DataFrame é retornado sem alterações.

**conform_to_schema**: Adapta o DataFrame de um arquivo `.parquet` ao `StructType` unificado definido para o dataset. Primeiro aplica `dedupe_airport_fee` para eliminar a duplicidade de grafia. Em seguida, para cada campo do schema alvo, converte a coluna existente para o tipo de dado definido (`cast`) ou, caso a coluna não exista no arquivo (ex: colunas criadas em anos posteriores), a cria preenchida com `NULL` no tipo esperado. Ao final, seleciona apenas as colunas do schema alvo, na ordem definida, descartando qualquer coluna fora dele (como `cbd_congestion_fee` em arquivos de 2025+, caso não conste no schema). O DataFrame resultante fica pronto para o `append` na tabela `bronze`.

In [0]:
def dedupe_airport_fee(df):
    has_lower = "airport_fee" in df.columns
    has_upper = "Airport_fee" in df.columns

    if has_lower and has_upper:
        df = df.withColumn("airport_fee_merged", coalesce(col("airport_fee"), col("Airport_fee")))
        df = df.drop("airport_fee", "Airport_fee")
        df = df.withColumnRenamed("airport_fee_merged", "airport_fee")
    elif has_upper and not has_lower:
        df = df.withColumnRenamed("Airport_fee", "airport_fee")

    return df

def conform_to_schema(df, schema):
    df = dedupe_airport_fee(df)
    for field in schema.fields:
        if field.name in df.columns:
            df = df.withColumn(field.name, col(field.name).cast(field.dataType))
        else:
            df = df.withColumn(field.name, lit(None).cast(field.dataType))
    return df.select([f.name for f in schema.fields])

### bronze.yellowTaxiTripData (Táxi Amarelo)

**Schema alvo (`target_schema`)**: 21 colunas, adotando o tipo mais permissivo encontrado no histórico de arquivos. Os campos `passenger_count` e `RatecodeID` foram definidos como `Double` (e não `Long`) por aparecerem com esse tipo em parte dos arquivos, e os timestamps foram definidos como `TimestampNTZType` (sem fuso horário). Inclui `cbd_congestion_fee` (taxa de congestionamento, presente apenas a partir de 2025) e a coluna de controle `_source_file_yearmonth`.

**Transformações e regras aplicadas:**

1. **Criação da tabela**: a tabela é criada vazia, já com o schema alvo, sobrescrevendo qualquer versão anterior (`overwrite`), o que garante que cada execução do notebook reprocesse a bronze do zero, sem duplicar linhas de execuções anteriores.
2. **Seleção dos arquivos**: são processados apenas os arquivos do volume com prefixo `yellow_tripdata_` e extensão `.parquet`.
3. **Coluna de rastreabilidade**: `_source_file_yearmonth` é extraída do nome do arquivo de origem (padrão `YYYY-MM`, via `_metadata.file_name`), permitindo rastrear a que arquivo mensal cada registro pertence.
4. **Conformação de schema**: cada arquivo passa pela função `conform_to_schema`, que resolve a duplicidade `airport_fee`/`Airport_fee`, converte cada coluna para o tipo do schema alvo, cria como `NULL` as colunas ausentes no arquivo e descarta colunas fora do schema.
5. **Tolerância a falhas por arquivo**: erros na leitura ou escrita de um arquivo são exibidos no log e o processamento continua com o arquivo seguinte, sem interromper a carga.
6. **Contagem de linhas**: o número de linhas de cada arquivo é registrado no log e somado ao total exibido ao final da execução.

**Comando final de escrita** (executado uma vez por arquivo, após a criação da tabela vazia):

```python
df.write.mode("append").saveAsTable("yellowTaxiTripData")
```

In [0]:
target_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
    StructField("_source_file_yearmonth", StringType(), True)
])

files = dbutils.fs.ls("/Volumes/mvp/staging/yellow_taxi")
parquet_files = [f for f in files if f.name.startswith("yellow_tripdata_") and f.name.endswith(".parquet")]
print(f"Encontrado {len(parquet_files)} arquivos para carregar")

total_rows = 0

empty_df = spark.createDataFrame([], target_schema)
empty_df.write.mode("overwrite").saveAsTable("yellowTaxiTripData")

for i, file_info in enumerate(parquet_files, 1):
    file_path = file_info.path
    file_name = file_info.name
    
    print(f"[{i}/{len(parquet_files)}] {file_name}...", end=" ")
    
    try:
        df = spark.read.parquet(file_path)

        df = df.withColumn(
            "_source_file_yearmonth",
            regexp_extract(col("_metadata.file_name"), r"(\d{4}-\d{2})", 1)
            )
        
        df = conform_to_schema(df, target_schema)
        
        row_count = df.count()
        total_rows += row_count
        
        df.write.mode("append").saveAsTable("yellowTaxiTripData")
        
        print(f"✓ {row_count:,} linhas")
    except Exception as e:
        print(f"❌ Erro: {str(e)}")
        continue

print(f"\n✅ Tabela Bronze criada com {total_rows:,} linhas totais!")

Encontrado 125 arquivos para carregar
[1/125] yellow_tripdata_2016-01.parquet... ✓ 10,905,067 linhas
[2/125] yellow_tripdata_2016-02.parquet... ✓ 11,375,412 linhas
[3/125] yellow_tripdata_2016-03.parquet... ✓ 12,203,824 linhas
[4/125] yellow_tripdata_2016-04.parquet... ✓ 11,927,996 linhas
[5/125] yellow_tripdata_2016-05.parquet... ✓ 11,832,049 linhas
[6/125] yellow_tripdata_2016-06.parquet... ✓ 11,131,645 linhas
[7/125] yellow_tripdata_2016-07.parquet... ✓ 10,294,080 linhas
[8/125] yellow_tripdata_2016-08.parquet... ✓ 9,942,263 linhas
[9/125] yellow_tripdata_2016-09.parquet... ✓ 10,116,018 linhas
[10/125] yellow_tripdata_2016-10.parquet... ✓ 10,854,626 linhas
[11/125] yellow_tripdata_2016-11.parquet... ✓ 10,102,128 linhas
[12/125] yellow_tripdata_2016-12.parquet... ✓ 10,446,697 linhas
[13/125] yellow_tripdata_2017-01.parquet... ✓ 9,710,820 linhas
[14/125] yellow_tripdata_2017-02.parquet... ✓ 9,169,775 linhas
[15/125] yellow_tripdata_2017-03.parquet... ✓ 10,295,441 linhas
[16/125] yello

### bronze.greenTaxiTripData (Táxi Verde)

**Schema alvo (`target_schema`)**: 21 colunas, adotando o tipo mais permissivo encontrado no histórico de arquivos. Os campos `passenger_count`, `RatecodeID`, `payment_type` e `trip_type` foram definidos como `Double` (e não `Long`) por aparecerem com esse tipo em parte dos arquivos, e os timestamps foram definidos como `TimestampNTZType` (sem fuso horário). Em relação ao táxi amarelo, os campos de data/hora usam o prefixo `lpep_` (em vez de `tpep_`), e o schema inclui `trip_type` e não inclui `airport_fee`. Inclui também `cbd_congestion_fee` (taxa de congestionamento, presente apenas a partir de 2025) e a coluna de controle `_source_file_yearmonth`.

**Transformações e regras aplicadas:**

1. **Criação da tabela**: a tabela é criada vazia, já com o schema alvo, sobrescrevendo qualquer versão anterior (`overwrite`), o que garante que cada execução do notebook reprocesse a bronze do zero, sem duplicar linhas de execuções anteriores.
2. **Seleção dos arquivos**: são processados apenas os arquivos do volume com prefixo `green_tripdata_` e extensão `.parquet`.
3. **Coluna de rastreabilidade**: `_source_file_yearmonth` é extraída do nome do arquivo de origem (padrão `YYYY-MM`, via `_metadata.file_name`), permitindo rastrear a que arquivo mensal cada registro pertence.
4. **Conformação de schema**: cada arquivo passa pela função `conform_to_schema`, que converte cada coluna para o tipo do schema alvo, cria como `NULL` as colunas ausentes no arquivo e descarta colunas fora do schema.
5. **Tolerância a falhas por arquivo**: erros na leitura ou escrita de um arquivo são exibidos no log e o processamento continua com o arquivo seguinte, sem interromper a carga.
6. **Contagem de linhas**: o número de linhas de cada arquivo é registrado no log e somado ao total exibido ao final da execução.

**Comando final de escrita** (executado uma vez por arquivo, após a criação da tabela vazia):

```python
df.write.mode("append").saveAsTable("greenTaxiTripData")
```

In [0]:
target_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("lpep_pickup_datetime", TimestampNTZType(), True),
    StructField("lpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("trip_type", DoubleType(), True),
    StructField("_source_file_yearmonth", StringType(), True)
])

files = dbutils.fs.ls("/Volumes/mvp/staging/green_taxi")
parquet_files = [f for f in files if f.name.startswith("green_tripdata_") and f.name.endswith(".parquet")]
print(f"Encontrado {len(parquet_files)} arquivos para carregar")

total_rows = 0

empty_df = spark.createDataFrame([], target_schema)
empty_df.write.mode("overwrite").saveAsTable("greenTaxiTripData")

for i, file_info in enumerate(parquet_files, 1):
    file_path = file_info.path
    file_name = file_info.name
    
    print(f"[{i}/{len(parquet_files)}] {file_name}...", end=" ")
    
    try:

        df = spark.read.parquet(file_path)

        df = df.withColumn(
            "_source_file_yearmonth",
            regexp_extract(col("_metadata.file_name"), r"(\d{4}-\d{2})", 1)
            )
        
        df = conform_to_schema(df, target_schema)
        
        row_count = df.count()
        total_rows += row_count
        
        df.write.mode("append").saveAsTable("greenTaxiTripData")
        
        print(f"✓ {row_count:,} linhas")
    except Exception as e:
        print(f"❌ Erro: {str(e)}")
        continue

print(f"\n✅ Tabela Bronze criada com {total_rows:,} linhas totais!")

Encontrado 126 arquivos para carregar
[1/126] green_tripdata_2016-01.parquet... ✓ 1,445,292 linhas
[2/126] green_tripdata_2016-02.parquet... ✓ 1,510,722 linhas
[3/126] green_tripdata_2016-03.parquet... ✓ 1,576,393 linhas
[4/126] green_tripdata_2016-04.parquet... ✓ 1,543,926 linhas
[5/126] green_tripdata_2016-05.parquet... ✓ 1,536,979 linhas
[6/126] green_tripdata_2016-06.parquet... ✓ 1,404,727 linhas
[7/126] green_tripdata_2016-07.parquet... ✓ 1,332,510 linhas
[8/126] green_tripdata_2016-08.parquet... ✓ 1,247,675 linhas
[9/126] green_tripdata_2016-09.parquet... ✓ 1,162,373 linhas
[10/126] green_tripdata_2016-10.parquet... ✓ 1,252,572 linhas
[11/126] green_tripdata_2016-11.parquet... ✓ 1,148,214 linhas
[12/126] green_tripdata_2016-12.parquet... ✓ 1,224,158 linhas
[13/126] green_tripdata_2017-01.parquet... ✓ 1,069,565 linhas
[14/126] green_tripdata_2017-02.parquet... ✓ 1,022,313 linhas
[15/126] green_tripdata_2017-03.parquet... ✓ 1,157,827 linhas
[16/126] green_tripdata_2017-04.parquet..

### bronze.forHireVehicles (For-Hire Vehicles, FHV)

**Schema alvo (`target_schema`)**: 8 colunas, adotando o tipo mais permissivo encontrado no histórico de arquivos. Os campos de localização (`PUlocationID`, `DOlocationID`) e `SR_Flag` foram definidos como `Double` (e não `Long`) por aparecerem com esse tipo em parte dos arquivos, e os timestamps foram definidos como `TimestampNTZType` (sem fuso horário). Em relação às bases de táxi amarelo e verde, a base FHV possui um schema bem mais enxuto: não contém dados de tarifa, distância, passageiros ou método de pagamento, apenas horários e locais de embarque/desembarque e o código da base de despacho (`dispatching_base_num`) e da base afiliada (`Affiliated_base_number`). Inclui também a coluna de controle `_source_file_yearmonth`.

**Transformações e regras aplicadas:**

1. **Criação da tabela**: a tabela é criada vazia, já com o schema alvo, sobrescrevendo qualquer versão anterior (`overwrite`), o que garante que cada execução do notebook reprocesse a bronze do zero, sem duplicar linhas de execuções anteriores.
2. **Seleção dos arquivos**: são processados apenas os arquivos do volume com prefixo `fhv` e extensão `.parquet`.
3. **Coluna de rastreabilidade**: `_source_file_yearmonth` é extraída do nome do arquivo de origem (padrão `YYYY-MM`, via `_metadata.file_name`), permitindo rastrear a que arquivo mensal cada registro pertence.
4. **Conformação de schema**: cada arquivo passa pela função `conform_to_schema`, que converte cada coluna para o tipo do schema alvo, cria como `NULL` as colunas ausentes no arquivo e descarta colunas fora do schema.
5. **Tolerância a falhas por arquivo**: erros na leitura ou escrita de um arquivo são exibidos no log e o processamento continua com o arquivo seguinte, sem interromper a carga.
6. **Contagem de linhas**: o número de linhas de cada arquivo é registrado no log e somado ao total exibido ao final da execução.

**Comando final de escrita** (executado uma vez por arquivo, após a criação da tabela vazia):

```python
df.write.mode("append").saveAsTable("forHireVehicles")
```

In [0]:
target_schema = StructType([
    StructField("Affiliated_base_number", StringType(), True),
    StructField("pickup_datetime", TimestampNTZType(), True),
    StructField("dropOff_datetime", TimestampNTZType(), True),
    StructField("DOlocationID", DoubleType(), True),
    StructField("PUlocationID", DoubleType(), True),
    StructField("SR_Flag", DoubleType(), True),
    StructField("dispatching_base_num", StringType(), True),
    StructField("_source_file_yearmonth", StringType(), True)
])

files = dbutils.fs.ls("/Volumes/mvp/staging/fhv")
parquet_files = [f for f in files if f.name.startswith("fhv") and f.name.endswith(".parquet")]
print(f"Encontrado {len(parquet_files)} arquivos para carregar")

total_rows = 0

empty_df = spark.createDataFrame([], target_schema)
empty_df.write.mode("overwrite").saveAsTable("forHireVehicles")

for i, file_info in enumerate(parquet_files, 1):
    file_path = file_info.path
    file_name = file_info.name
    
    print(f"[{i}/{len(parquet_files)}] {file_name}...", end=" ")
    
    try:

        df = spark.read.parquet(file_path)

        df = df.withColumn(
            "_source_file_yearmonth",
            regexp_extract(col("_metadata.file_name"), r"(\d{4}-\d{2})", 1)
            )
        
        df = conform_to_schema(df, target_schema)
        
        row_count = df.count()
        total_rows += row_count
        
        df.write.mode("append").saveAsTable("forHireVehicles")
        
        print(f"✓ {row_count:,} linhas")
    except Exception as e:
        print(f"❌ Erro: {str(e)}")
        continue

print(f"\n✅ Tabela Bronze criada com {total_rows:,} linhas totais!")

Encontrado 124 arquivos para carregar
[1/124] fhv_tripdata_2016-01.parquet... ✓ 8,510,803 linhas
[2/124] fhv_tripdata_2016-02.parquet... ✓ 9,427,825 linhas
[3/124] fhv_tripdata_2016-03.parquet... ✓ 9,613,395 linhas
[4/124] fhv_tripdata_2016-04.parquet... ✓ 10,138,188 linhas
[5/124] fhv_tripdata_2016-05.parquet... ✓ 10,739,532 linhas
[6/124] fhv_tripdata_2016-06.parquet... ✓ 10,768,623 linhas
[7/124] fhv_tripdata_2016-07.parquet... ✓ 10,977,358 linhas
[8/124] fhv_tripdata_2016-08.parquet... ✓ 11,224,963 linhas
[9/124] fhv_tripdata_2016-09.parquet... ✓ 11,759,658 linhas
[10/124] fhv_tripdata_2016-10.parquet... ✓ 12,657,985 linhas
[11/124] fhv_tripdata_2016-11.parquet... ✓ 12,436,993 linhas
[12/124] fhv_tripdata_2016-12.parquet... ✓ 13,858,760 linhas
[13/124] fhv_tripdata_2017-01.parquet... ✓ 13,657,539 linhas
[14/124] fhv_tripdata_2017-02.parquet... ✓ 13,284,960 linhas
[15/124] fhv_tripdata_2017-03.parquet... ✓ 15,703,098 linhas
[16/124] fhv_tripdata_2017-04.parquet... ✓ 14,764,943 linha

### bronze.forHireVehiclesHighVolume (High Volume For-Hire Services, HVFHS)

**Schema alvo (`target_schema`)**: 26 colunas. Diferente das bases de táxi, a base HVFHS registra o ciclo completo da viagem por aplicativo (`request_datetime`, `on_scene_datetime`, `pickup_datetime`, `dropoff_datetime`), a licença da operadora (`hvfhs_license_num`), as bases de despacho e origem (`dispatching_base_num`, `originating_base_num`), além de componentes detalhados de tarifa e remuneração: `base_passenger_fare` (tarifa base ao passageiro), `driver_pay` (remuneração do motorista), `tips`, `tolls`, `bcf`, `sales_tax`, `congestion_surcharge`, `airport_fee` e `cbd_congestion_fee` (presente apenas a partir de 2025). Também possui flags de viagem compartilhada (`shared_request_flag`, `shared_match_flag`), de acessibilidade (`wav_request_flag`, `wav_match_flag`, `access_a_ride_flag`) e os campos de distância e duração (`trip_miles`, `trip_time`). Os timestamps foram definidos como `TimestampNTZType` (sem fuso horário). Inclui também a coluna de controle `_source_file_yearmonth`.

**Transformações e regras aplicadas:**

1. **Criação da tabela**: a tabela é criada vazia, já com o schema alvo, sobrescrevendo qualquer versão anterior (`overwrite`), o que garante que cada execução do notebook reprocesse a bronze do zero, sem duplicar linhas de execuções anteriores.
2. **Seleção dos arquivos**: são processados apenas os arquivos do volume com prefixo `fhvhv` e extensão `.parquet`.
3. **Coluna de rastreabilidade**: `_source_file_yearmonth` é extraída do nome do arquivo de origem (padrão `YYYY-MM`, via `_metadata.file_name`), permitindo rastrear a que arquivo mensal cada registro pertence.
4. **Conformação de schema**: cada arquivo passa pela função `conform_to_schema`, que resolve a duplicidade de grafia `airport_fee`/`Airport_fee` (presente nos arquivos de 2025 em diante), converte cada coluna para o tipo do schema alvo, cria como `NULL` as colunas ausentes no arquivo e descarta colunas fora do schema.
5. **Tolerância a falhas por arquivo**: erros na leitura ou escrita de um arquivo são exibidos no log e o processamento continua com o arquivo seguinte, sem interromper a carga.
6. **Contagem de linhas**: o número de linhas de cada arquivo é registrado no log e somado ao total exibido ao final da execução.

**Comando final de escrita** (executado uma vez por arquivo, após a criação da tabela vazia):

```python
df.write.mode("append").saveAsTable("forHireVehiclesHighVolume")
```

In [0]:
target_schema = StructType([
    StructField("originating_base_num", StringType(), True),
    StructField("dispatching_base_num", StringType(), True),
    StructField("request_datetime", TimestampNTZType(), True),
    StructField("on_scene_datetime", TimestampNTZType(), True),
    StructField("pickup_datetime", TimestampNTZType(), True),
    StructField("dropoff_datetime", TimestampNTZType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("access_a_ride_flag", StringType(), True),
    StructField("airport_fee", DoubleType(), True),
    StructField("base_passenger_fare", DoubleType(), True),
    StructField("bcf", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("driver_pay", DoubleType(), True),
    StructField("hvfhs_license_num", StringType(), True),
    StructField("sales_tax", DoubleType(), True),
    StructField("shared_match_flag", StringType(), True),
    StructField("shared_request_flag", StringType(), True),
    StructField("tips", DoubleType(), True),
    StructField("tolls", DoubleType(), True),
    StructField("trip_miles", DoubleType(), True),
    StructField("trip_time", LongType(), True),
    StructField("wav_match_flag", StringType(), True),
    StructField("wav_request_flag", StringType(), True),
    StructField("_source_file_yearmonth", StringType(), True)
])

files = dbutils.fs.ls("/Volumes/mvp/staging/fhvhv")
parquet_files = [f for f in files if f.name.startswith("fhvhv") and f.name.endswith(".parquet")]
print(f"Encontrado {len(parquet_files)} arquivos para carregar")

total_rows = 0

empty_df = spark.createDataFrame([], target_schema)
empty_df.write.mode("overwrite").saveAsTable("forHireVehiclesHighVolume")

for i, file_info in enumerate(parquet_files, 1):
    file_path = file_info.path
    file_name = file_info.name
    
    print(f"[{i}/{len(parquet_files)}] {file_name}...", end=" ")
    
    try:
        df = spark.read.parquet(file_path)

        df = df.withColumn(
            "_source_file_yearmonth",
            regexp_extract(col("_metadata.file_name"), r"(\d{4}-\d{2})", 1)
            )
        
        df = conform_to_schema(df, target_schema)
        
        row_count = df.count()
        total_rows += row_count
        
        df.write.mode("append").saveAsTable("forHireVehiclesHighVolume")
        
        print(f"✓ {row_count:,} linhas")
    except Exception as e:
        print(f"❌ Erro: {str(e)}")
        continue

print(f"\n✅ Tabela Bronze criada com {total_rows:,} linhas totais!")

Encontrado 89 arquivos para carregar
[1/89] fhvhv_tripdata_2019-02.parquet... ✓ 20,159,102 linhas
[2/89] fhvhv_tripdata_2019-03.parquet... ✓ 23,864,598 linhas
[3/89] fhvhv_tripdata_2019-04.parquet... ✓ 21,734,822 linhas
[4/89] fhvhv_tripdata_2019-05.parquet... ✓ 22,329,247 linhas
[5/89] fhvhv_tripdata_2019-06.parquet... ✓ 21,001,990 linhas
[6/89] fhvhv_tripdata_2019-07.parquet... ✓ 20,303,312 linhas
[7/89] fhvhv_tripdata_2019-08.parquet... ✓ 20,126,113 linhas
[8/89] fhvhv_tripdata_2019-09.parquet... ✓ 20,069,321 linhas
[9/89] fhvhv_tripdata_2019-10.parquet... ✓ 21,162,290 linhas
[10/89] fhvhv_tripdata_2019-11.parquet... ✓ 21,635,568 linhas
[11/89] fhvhv_tripdata_2019-12.parquet... ✓ 22,243,901 linhas
[12/89] fhvhv_tripdata_2020-01.parquet... ✓ 20,569,368 linhas
[13/89] fhvhv_tripdata_2020-02.parquet... ✓ 21,725,100 linhas
[14/89] fhvhv_tripdata_2020-03.parquet... ✓ 13,392,928 linhas
[15/89] fhvhv_tripdata_2020-04.parquet... ✓ 4,312,909 linhas
[16/89] fhvhv_tripdata_2020-05.parquet... ✓

### bronze.taxi_zone_lookup (Zonas de Táxi)

**Transformações e regras aplicadas:**

1. **Leitura do CSV**: o arquivo é lido com cabeçalho (`header=True`), e o schema é inferido automaticamente a partir dos dados (`inferSchema=True`).
2. **Sem transformações de conteúdo**: os dados são gravados como recebidos, seguindo o princípio da camada bronze (dado bruto, sem tratamento).
3. **Criação da tabela**: a tabela é gravada em modo `overwrite`, substituindo qualquer versão anterior a cada execução.

**Comando final de escrita:**

```python
csv.write.mode("overwrite").saveAsTable("taxi_zone_lookup")
```

In [0]:
file = dbutils.fs.ls("/Volumes/mvp/staging/taxi_zones/taxi_zone_lookup.csv")

csv = spark.read.csv(file[0].path, header=True, inferSchema=True)
csv.write.mode("overwrite").saveAsTable("taxi_zone_lookup")

### bronze.fhv_base_lookup (Bases de Despacho FHV / Operadoras)

**Transformações e regras aplicadas:**

1. **Leitura do CSV**: o arquivo é lido com cabeçalho (`header=True`), separador ponto e vírgula (`sep=";"`, padrão de exportação do Excel em configuração regional brasileira) e schema inferido automaticamente (`inferSchema=True`). Por se tratar de uma tabela pequena e mantida manualmente, não foi necessário definir um `StructType` manual como nas bases de viagens.
2. **Sem transformações de conteúdo**: os dados são gravados como recebidos, seguindo o princípio da camada bronze.
3. **Criação da tabela**: a tabela é gravada em modo `overwrite`, substituindo qualquer versão anterior a cada execução.

**Comando final de escrita:**

```python
csv.write.mode("overwrite").saveAsTable("fhv_base_lookup")
```

In [0]:
file = dbutils.fs.ls("/Volumes/mvp/staging/misc/fhv_lookup.csv")

csv = spark.read.csv(file[0].path, header=True, inferSchema=True, sep=";")
csv.write.mode("overwrite").saveAsTable("fhv_base_lookup")

### bronze.Feriados_US_NY (Feriados do Estado de Nova Iorque)

**Transformações e regras aplicadas:**

1. **Leitura do CSV**: o arquivo é lido com cabeçalho (`header=True`), separador vírgula (`sep=","`) e schema inferido automaticamente (`inferSchema=True`). Por se tratar de uma tabela pequena, não foi necessário definir um `StructType` manual como nas bases de viagens.
2. **Sem transformações de conteúdo**: os dados são gravados como recebidos, seguindo o princípio da camada bronze. O tratamento de feriados com data oficial e data de observância distintas é feito posteriormente, na camada `silver`, conforme descrito na seção de Modelagem.
3. **Criação da tabela**: a tabela é gravada em modo `overwrite`, substituindo qualquer versão anterior a cada execução.

**Comando final de escrita:**

```python
csv.write.mode("overwrite").saveAsTable("Feriados_US_NY")
```

In [0]:
file = dbutils.fs.ls("/Volumes/mvp/staging/misc/Feriados_US_NY.csv")

csv = spark.read.csv(file[0].path, header=True, inferSchema=True, sep=",")
csv.write.mode("overwrite").saveAsTable("Feriados_US_NY")

### bronze.Inflacao_CPI_U e bronze.Inflacao_CPI_T (Índices de Inflação)

**Transformações e regras aplicadas (idênticas para as duas tabelas):**

1. **Leitura do CSV**: cada arquivo é lido com cabeçalho (`header=True`), separador vírgula (`sep=","`) e schema inferido automaticamente (`inferSchema=True`). Por se tratar de séries pequenas, não foi necessário definir um `StructType` manual como nas bases de viagens.
2. **Sem transformações de conteúdo**: os dados são gravados como recebidos, seguindo o princípio da camada bronze.
3. **Criação das tabelas**: cada tabela é gravada em modo `overwrite`, substituindo qualquer versão anterior a cada execução.

**Comandos finais de escrita:**

```python
csv_cpi_u.write.mode("overwrite").saveAsTable("Inflacao_CPI_U")
csv_cpi_t.write.mode("overwrite").saveAsTable("Inflacao_CPI_T")
```

In [0]:
file = dbutils.fs.ls("/Volumes/mvp/staging/misc/CPIAUCSL (2016 - 2026).csv")

csv = spark.read.csv(file[0].path, header=True, inferSchema=True, sep=",")
csv.write.mode("overwrite").saveAsTable("Inflacao_CPI_U")

In [0]:
file = dbutils.fs.ls("/Volumes/mvp/staging/misc/CPITRNSL (20216-2026).csv")

csv = spark.read.csv(file[0].path, header=True, inferSchema=True, sep=",")
csv.write.mode("overwrite").saveAsTable("Inflacao_CPI_T")